#imports

In [102]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.tree import plot_tree
from matplotlib.pylab import rcParams
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
df = pd.read_csv('/content/Social_Network_Ads.csv')
df.head()

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0


#data preprocessing

In [103]:
df['Gender']=df['Gender'].replace({'Male':1,'Female':0})
df.head()

/tmp/ipykernel_512/3665408947.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Gender']=df['Gender'].replace({'Male':1,'Female':0})


,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,1,19,19000,0
1,15810944,1,35,20000,0
2,15668575,0,26,43000,0
3,15603246,0,27,57000,0
4,15804002,1,19,76000,0


In [104]:
X = df.iloc[:,0:4]
y = df.iloc[:,-1]

#train test split

In [105]:
x_train,x_test,y_train,y_test = train_test_split(X,y,random_state=2,test_size=0.2)

#decision tree object
#<font color='red'>max_depth = 4 <br>critrion=gini<br>splitter=random</font>


In [106]:
dtc = DecisionTreeClassifier(max_depth=4,criterion='gini',splitter='random',random_state=2)

#fit

In [107]:
dtc.fit(x_train,y_train)

DecisionTreeClassifier(max_depth=4, random_state=2, splitter='random')

#predict

In [108]:
y_pred = dtc.predict(x_test)

#calculating error

In [109]:
mean_squared_error(y_pred,y_test)

0.175

#<font color='red'>setting-hyperparameter with GridSearchCV

In [110]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Decision Tree model

In [111]:
dtc = DecisionTreeClassifier(random_state=2)

# Hyperparameter combinations

In [112]:
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 2, 4, 6, 8, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

# Grid Search

In [113]:
grid_search = GridSearchCV(
    estimator=dtc,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Training

In [114]:
grid_search.fit(x_train, y_train)

Fitting 5 folds for each of 972 candidates, totalling 4860 fits


GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=2), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [None, 2, 4, 6, 8, 10],
                         'max_features': [None, 'sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'splitter': ['best', 'random']},
             scoring='accuracy', verbose=1)

# Best parameters

In [115]:
print("Best CV Accuracy:", grid_search.best_score_)

Best CV Accuracy: 0.90625


# Best model

In [116]:
best_dtc_grid = grid_search.best_estimator_

# Testing

In [117]:
y_pred_grid = best_dtc_grid.predict(x_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_grid))
print(classification_report(y_test, y_pred_grid))

Test Accuracy: 0.8875
              precision    recall  f1-score   support

           0       0.88      0.94      0.91        48
           1       0.90      0.81      0.85        32

    accuracy                           0.89        80
   macro avg       0.89      0.88      0.88        80
weighted avg       0.89      0.89      0.89        80



#<font color='red'>setting-hyperparameter with RandomSearchCV

In [118]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from scipy.stats import randint

# Decision Tree model
dtc = DecisionTreeClassifier(random_state=2)

# Hyperparameter distributions
param_dist = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 2, 4, 6, 8, 10, 15, 20],
    'min_samples_split': randint(2, 21),
    'min_samples_leaf': randint(1, 11),
    'max_features': [None, 'sqrt', 'log2']
}

# Randomized Search
random_search = RandomizedSearchCV(
    estimator=dtc,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=2,
    verbose=1
)

# Training
random_search.fit(x_train, y_train)

# Best parameters
print("Best Parameters:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)

# Best model
best_dtc_random = random_search.best_estimator_

# Testing
y_pred_random = best_dtc_random.predict(x_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_random))
print(classification_report(y_test, y_pred_random))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'criterion': 'entropy', 'max_depth': 4, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 2, 'splitter': 'best'}
Best CV Accuracy: 0.90625
Test Accuracy: 0.8875
              precision    recall  f1-score   support

           0       0.88      0.94      0.91        48
           1       0.90      0.81      0.85        32

    accuracy                           0.89        80
   macro avg       0.89      0.88      0.88        80
weighted avg       0.89      0.89      0.89        80

